In [ ]:
!pip install -U imageio imageio-ffmpeg tqdm
!pip install -U kaleido


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 13.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import re
import json

# Load your TSV dataset
df = pd.read_csv("haunted_places_v2.tsv", sep="\t")

# Function to extract the first 4-digit year from the description
def extract_year(text):
    match = re.search(r'\b(18[0-9]{2}|19[0-9]{2}|20[0-2][0-9]|2025)\b', str(text))
    return int(match.group()) if match else None

# Create 'year' column
df["year"] = df["description"].apply(extract_year)

# Drop rows where year couldn't be extracted
df_clean = df.dropna(subset=["year"])

# Count occurrences of city per year
city_year_counts = df_clean.groupby(["year", "city"]).size().reset_index(name="count")

# Pivot the table to have years as rows and cities as columns
pivot_df = city_year_counts.pivot(index="year", columns="city", values="count").fillna(0).astype(int)

# Optional: sort cities alphabetically for consistency
pivot_df = pivot_df.reindex(sorted(pivot_df.columns), axis=1)

# Convert pivoted DataFrame to list of records (dicts)
bar_race_data = pivot_df.reset_index().to_dict(orient="records")

# Save to JSON
with open("haunted_bar_chart_race.json", "w") as f:
    json.dump(bar_race_data, f, indent=2)

print("JSON file 'haunted_bar_chart_race.json' created successfully!")


✅ JSON file 'haunted_bar_chart_race.json' created successfully!


In [ ]:
import pandas as pd
import json
from tqdm.notebook import tqdm
import plotly.express as px
import os
import imageio

# Load JSON
with open("haunted_bar_chart_race.json", "r") as f:
    data = json.load(f)

df = pd.DataFrame(data)
df_long = df.melt(id_vars="year", var_name="city", value_name="count")
df_long['year'] = df_long['year'].astype(int)


In [ ]:
# Get top 10 cities with most total hauntings
top_cities = (
    df_long.groupby("city")["count"].sum()
    .sort_values(ascending=False)
    .head(10)
    .index.tolist()
)

# Filter to those cities
df_top10 = df_long[df_long["city"].isin(top_cities)].copy()

# Update the city list with emoji version
top_cities_emoji = df_top10["city"].unique().tolist()

# Create full year range and fill in missing years with 0
all_years = sorted(df_top10["year"].unique())
full_data = pd.DataFrame([(year, city) for year in all_years for city in top_cities_emoji], columns=["year", "city"])
df_top10_full = pd.merge(full_data, df_top10, on=["year", "city"], how="left").fillna(0)
df_top10_full["count"] = df_top10_full["count"].astype(int)

# Compute cumulative sum
df_top10_full["cumulative"] = df_top10_full.sort_values("year").groupby("city")["count"].cumsum()


In [ ]:
df_top10_fixed = df_long[df_long["city"].isin(top_cities)].copy()
df_top10_fixed["city"] = df_top10_fixed["city"]
years = sorted(df_top10_fixed["year"].unique())

# Create folder for frames
os.makedirs("frames", exist_ok=True)
max_count = df_top10_fixed["count"].max()


In [ ]:
from tqdm.notebook import tqdm
import plotly.express as px
import pandas as pd

# Ensure folders exist
os.makedirs("frames", exist_ok=True)

# Max value for consistent scale
max_count = df_top10_full["cumulative"].max()

print("📷 Saving smoother chart frames with labels...")
for year in tqdm(all_years):
    df_frame = df_top10_full[df_top10_full["year"] == year]

    fig = px.bar(
        df_frame.sort_values("cumulative", ascending=True),
        x="cumulative",
        y="city",
        color="city",
        orientation="h",
        range_x=[0, max_count + 10],
        title=f"👻 Haunted Places Growth\nTop 10 Cities by Cumulative Sightings\nYear: {year}",
        height=600,
        text="cumulative"
    )

    fig.update_traces(
        texttemplate='%{text}',
        textposition='outside',
        marker_line_width=0.5
    )

    fig.update_layout(
        yaxis=dict(categoryorder="total ascending"),
        showlegend=False,
        plot_bgcolor="black",
        paper_bgcolor="black",
        font=dict(color="white", size=16),
        margin=dict(l=100, r=40, t=100, b=40),
        title_font_size=24,
        uniformtext_minsize=12,
        uniformtext_mode='hide'
    )

    # Save smoother frame
    fig.write_image(f"frames/frame_{year}.png", engine="kaleido")


📷 Saving smoother chart frames with labels...


  0%|          | 0/182 [00:00<?, ?it/s]

In [ ]:
print("🎞️ Creating  MP4 at 10 FPS...")
with imageio.get_writer("haunted_top10_cumulative_smooth.mp4", fps=10) as writer:
    for year in all_years:
        image = imageio.v2.imread(f"frames/frame_{year}.png")
        writer.append_data(image)

print("✅ Saved as haunted_top10_cumulative_smooth.mp4")


🎞️ Creating  MP4 at 10 FPS...


✅ Saved as haunted_top10_cumulative_smooth.mp4


In [ ]:
from moviepy.editor import VideoFileClip

# Load the video
video = VideoFileClip("haunted_top10_cumulative_smooth.mp4")

# Speed up the video by a factor of 2
sped_up_video = video.fx(vfx.speedx, 2)

# Write the output video
sped_up_video.write_videofile("haunted_top10_cumulative_smooth_speedup.mp4")


NameError: name 'vfx' is not defined

In [ ]:
import pandas as pd
import json
import numpy as np
import os
from tqdm.notebook import tqdm
import plotly.express as px
import imageio

# Load JSON again if needed
with open("haunted_bar_chart_race.json", "r") as f:
    data = json.load(f)

df = pd.DataFrame(data)
df_long = df.melt(id_vars="year", var_name="location", value_name="count")
df_long["year"] = df_long["year"].astype(int)

# Get mapping from city to state using original TSV if needed
# But assuming now that we have a state version of df with state info
# We'll simulate it using city-to-state assumption next


In [ ]:
# Load full TSV file that includes state info
df_places = pd.read_csv("haunted_places_v2.tsv", sep="\t")

# Build a mapping of city to state
city_to_state = df_places.set_index("city")["state"].to_dict()

# Add state column to df_long
df_long["state"] = df_long["location"].map(city_to_state)
df_long = df_long.dropna(subset=["state"])


In [ ]:
# Get total count per state
top_states = (
    df_long.groupby("state")["count"].sum()
    .sort_values(ascending=False)
    .head(10)
    .index.tolist()
)

# Filter to those
df_state_top10 = df_long[df_long["state"].isin(top_states)].copy()

# Create full (year, state) grid and fill missing years
all_years = sorted(df_state_top10["year"].unique())
grid = pd.DataFrame([(y, s) for y in all_years for s in top_states], columns=["year", "state"])
df_full = pd.merge(grid, df_state_top10, on=["year", "state"], how="left").fillna(0)
df_full["count"] = df_full["count"].astype(int)

# Compute cumulative count
df_full["cumulative"] = df_full.sort_values("year").groupby("state")["count"].cumsum()


In [ ]:
# Aggregate in case there are duplicates (shouldn't be after this)
df_grouped = df_full.groupby(["year", "state"], as_index=False)["cumulative"].sum()

# Pivot and interpolate
pivot = df_grouped.pivot(index="year", columns="state", values="cumulative").fillna(0)


# Define smooth years
steps_per_year = 3
real_years = sorted(df_full["year"].unique())
smooth_years = []
for i in range(len(real_years) - 1):
    smooth_years.extend(np.linspace(real_years[i], real_years[i+1], steps_per_year + 1)[:-1])
smooth_years.append(real_years[-1])

# Interpolate
pivot_interp = pivot.reindex(smooth_years).interpolate(method="linear")
df_interp = pivot_interp.reset_index().melt(id_vars="year", var_name="state", value_name="cumulative")


In [ ]:
os.makedirs("frames_state", exist_ok=True)
max_val = df_interp["cumulative"].max()

print("📷 Saving smooth state frames...")
for year in tqdm(sorted(df_interp["year"].unique())):
    df_frame = df_interp[df_interp["year"] == year]

    fig = px.bar(
        df_frame.sort_values("cumulative", ascending=True),
        x="cumulative",
        y="state",
        color="state",
        orientation="h",
        range_x=[0, max_val + 10],
        title=f"🗺️ Most Haunted U.S. States Over Time<br><span style='font-size:32px;'>Year: {int(round(year))}</span>",
        height=600,
        text="cumulative"
    )

    fig.update_traces(
        texttemplate='%{text:.0f}',
        textposition='outside',
        marker_line_width=0.5
    )

    fig.update_layout(
        yaxis=dict(categoryorder="total ascending"),
        showlegend=False,
        plot_bgcolor="black",
        paper_bgcolor="black",
        font=dict(color="white", size=16),
        title_font_size=24,
        margin=dict(l=100, r=40, t=100, b=40)
    )

    fig.write_image(f"frames_state/frame_{year:.2f}.png", engine="kaleido")


📷 Saving smooth state frames...


  0%|          | 0/544 [00:00<?, ?it/s]

In [ ]:
with imageio.get_writer("haunted_states_smooth.mp4", fps=30) as writer:
    for year in sorted(df_interp["year"].unique()):
        image = imageio.v2.imread(f"frames_state/frame_{year:.2f}.png")
        writer.append_data(image)

In [ ]:
import pandas as pd

# Load the uploaded TSV file
file_path = 'haunted_places_v2.tsv'
df = pd.read_csv(file_path, sep='\t')

# Show the first few rows of the dataset to understand its structure
df.head()


,city,country,description,location,state,state_abbrev,longitude,latitude,city_longitude,city_latitude,...,moon_distance,geotopic_name,geotopic_longitude,geotopic_latitude,NER_ENTITIES,NER_PERSONS,NER_ORGS,NER_LOCATIONS,image_path,tika_caption
0,Ada,United States,Ada witch - Sometimes you can see a misty blue...,Ada Cemetery,Michigan,MI,-85.504893,42.962106,-85.495480,42.960727,...,379317,Little Egypt Valley,-123.22210,47.22898,"[('Ada witch -', 'ORG'), ('3-mile', 'QUANTITY'...",['Findlay Cemetery'],"['Ada witch -', 'Egypt Valley']","['the Ada Cemetery', 'Honey Creek', 'Honeycree...",images/ada_michigan_0.png,"{""captions"": [{""confidence"": 0.000134258645026..."
1,Addison,United States,A little girl was killed suddenly while waitin...,North Adams Rd.,Michigan,MI,-84.381843,41.971425,-84.347168,41.986434,...,379317,Michigan,-85.50033,44.25029,"[('in.1 month later', 'DATE'), ('this day', 'D...",[],[],[],images/addison_michigan_1.png,"{""captions"": [{""confidence"": 8.482576685112755..."
2,Adrian,United States,If you take Gorman Rd. west towards Sand Creek...,Ghost Trestle,Michigan,MI,-84.035656,41.904538,-84.037166,41.897547,...,379317,Michigan,-85.50033,44.25029,"[('Gorman Rd', 'FAC'), ('Sand Creek', 'GPE'), ...",[],[],"['Gorman Rd', 'Sand Creek']",images/adrian_michigan_2.png,"{""captions"": [{""confidence"": 0.000180955290961..."
3,Adrian,United States,"In the 1970's, one room, room 211, in the old ...",Siena Heights University,Michigan,MI,-84.017565,41.905712,-84.037166,41.897547,...,379317,Michigan,-85.50033,44.25029,"[('1970', 'DATE'), ('one', 'CARDINAL'), ('211'...",[],[],[],images/adrian_michigan_3.png,"{""captions"": [{""confidence"": 3.237535527872346..."
4,Albion,United States,Kappa Delta Sorority - The Kappa Delta Sororit...,Albion College,Michigan,MI,-84.745177,42.244006,-84.753030,42.243097,...,379317,Michigan,-85.50033,44.25029,[('Kappa Delta Sorority - The Kappa Delta Soro...,[],['Kappa Delta Sorority - The Kappa Delta Soror...,[],images/albion_michigan_4.png,"{""captions"": [{""confidence"": 0.000302628798393..."


,city,description,mentions_binge_drink,mentions_alcohol_deaths,mentions_annual_deaths


,state,Over 18 binge drink at least once per month
0,Alabama,0.136
1,Alaska,0.160
2,Arizona,0.167
3,Arkansas,0.160
4,California,0.166


MessageError: Error: credential propagation was unsuccessful

{"Alabama": 0.136, "Alaska": 0.16, "Arizona": 0.16699999999999998, "Arkansas": 0.16, "California": 0.166, "Colorado": 0.198, "Connecticut": 0.187, "Delaware": 0.183, "Florida": 0.175, "Georgia": 0.151, "Hawaii": 0.209, "Idaho": 0.16, "Illinois": 0.205, "Indiana": 0.171, "Iowa": 0.245, "Kansas": 0.16699999999999998, "Kentucky": 0.158, "Louisiana": 0.181, "Maine": 0.195, "Maryland": 0.145, "Massachusetts": 0.21100000000000002, "Michigan": 0.198, "Minnesota": 0.21699999999999997, "Mississippi": 0.13699999999999998, "Missouri": 0.18899999999999997, "Montana": 0.19699999999999998, "Nebraska": 0.225, "Nevada": 0.16, "New Hampshire": 0.183, "New Jersey": 0.14400000000000002, "New Mexico": 0.156, "New York": 0.17600000000000002, "North Carolina": 0.158, "North Dakota": 0.22699999999999998, "Ohio": 0.172, "Oklahoma": 0.136, "Oregon": 0.17, "Pennsylvania": 0.185, "Rhode Island": 0.17800000000000002, "South Carolina": 0.163, "South Dakota": 0.22300000000000003, "Tennessee": 0.157, "Texas": 0.1760

SyntaxError: f-string: invalid syntax (<ipython-input-36-105732eecd15>, line 46)

Data saved to alcohol_data.json
